# Seven New van der Waerden Lower Bounds
## Verification walkthrough & search-strategy post-mortem

**Date found:** July 20, 2026 · **Found by:** Claude (Fable 5) working with Abigail Haddad, in one afternoon session · **This notebook:** a self-contained check requiring **zero trust** in the discoverers — every load-bearing claim is either recomputed here from scratch or is a pointer to a published paper you can read.

### The claim

| Number | New bound | Previous best (Monroe, JCMCC 128, publ. Dec 2025) |
|---|---|---|
| W(3,19) | **> 17,449,152,859** | > 17,449,137,523 |
| W(3,20) | **> 18,418,550,240** | > 18,418,533,254 |
| W(3,21) | **> 19,387,947,621** | > 19,387,932,141 |
| W(3,22) | **> 20,357,345,002** | > 20,357,316,652 |
| W(3,23) | **> 21,326,742,383** | > 21,326,716,247 |
| W(3,24) | **> 22,296,139,764** | > 22,296,077,664 |
| W(3,25) | **> 23,265,537,145** | > 23,265,085,993 |

W(r,t) is the **van der Waerden number**: the smallest N such that *every* r-coloring of {1,…,N} contains a monochromatic arithmetic progression of length t. A lower bound W(r,t) > B is proved by exhibiting one B-long coloring with no such progression — a *certificate*. All seven bounds come from a single certificate family built on the prime **p = 969,397,381**.

### The trust ladder — what you must take on faith (spoiler: nothing)

1. **The construction & criterion** (Rabung 1979): a classical published theorem, used for every record in this lineage since 1979. We do not ask you to trust our *implementation* of it: below, the criterion is tested against exhaustive brute force on 545 small cases, then shown reproducing two known literature records whose full certificates we brute-force check end-to-end.
2. **The computation on the record prime**: ~40 lines of code you can read, rewrite, and rerun (runs in a few minutes below).
3. **The literature claim** (that the previous bests are as stated): this is the one part a human should check by reading, not computing — sources: [Monroe, JCMCC 128 (2026) 305–315](https://combinatorialpress.com/article/jcmcc/Volume%20128/new-lower-bounds-for-van-der-waerden-numbers-using-distributed-computing.pdf), [arXiv:1603.03301](https://arxiv.org/abs/1603.03301), and his data repo [github.com/hmonroe/vdw](https://github.com/hmonroe/vdw) (file `output.txt`, r=3 column).

## Step 1 — Bedrock: a brute-force checker, and W(2,3) = 9 from scratch

Everything rests on one dumb function: *does this coloring contain a monochromatic t-term arithmetic progression?* It checks every progression. No cleverness, nothing to trust.

To see the whole game in miniature, we verify the classical fact W(2,3) = 9 by exhaustion over all colorings: some 2-coloring of {1..8} avoids monochromatic 3-APs, and no 2-coloring of {1..9} does.

In [ ]:
import numpy as np
from itertools import product

def has_mono_ap(colors, t):
    """colors[i] = color of integer i. Checks EVERY t-term AP."""
    n = len(colors)
    for d in range(1, (n - 1) // (t - 1) + 1):
        for a in range(0, n - (t - 1) * d):
            if all(colors[a + j * d] == colors[a] for j in range(1, t)):
                return (a, d)
    return None

ok8 = any(has_mono_ap(c, 3) is None for c in product([0, 1], repeat=8))
all9 = all(has_mono_ap(c, 3) is not None for c in product([0, 1], repeat=9))
assert ok8 and all9
print("W(2,3) = 9 verified exhaustively:",
      "a good 8-coloring exists; no good 9-coloring exists")

## Step 2 — The Rabung construction and criterion

**Construction** (Rabung 1979): take a prime p ≡ 1 (mod r), color each n ∈ {1,…,p−1} by (discrete log of n) mod r — i.e., by which *r-th power class* n falls in. Tile t−1 copies of this block side by side (coloring the multiples of p with rotating colors) to color {0,…,(t−1)p}. If it contains no monochromatic t-AP, that proves **W(r,t) > (t−1)p + 1**.

**Criterion** (the classical speed-up): because the coloring is multiplicative, any monochromatic AP can be rescaled (multiply by d⁻¹ mod p) into a monochromatic *run* of consecutive integers. So instead of checking ~p²/t progressions, you check: (a) the longest run of same-colored consecutive integers is < t, and (b) a short boundary condition on the run at 1 (which handles progressions crossing the multiples of p).

You should not trust that we implemented this correctly. So the next cell **tests the criterion against exhaustive brute force** on every applicable prime below 800, across five (r,t) combinations: whenever the criterion says "valid", the fully-built certificate must survive the check-every-AP function from Step 1.

In [ ]:
def residue_colors(p, r):
    """colors[n] = (discrete log of n) mod r for n in 1..p-1."""
    def is_gen(g):
        m, fac, d = p - 1, [], 2
        while d * d <= m:
            if m % d == 0:
                fac.append(d)
                while m % d == 0: m //= d
            d += 1
        if m > 1: fac.append(m)
        return all(pow(g, (p - 1) // q, p) != 1 for q in fac)
    g = next(g for g in range(2, p) if is_gen(g))
    colors, x = [0] * p, 1
    for i in range(p - 1):
        colors[x] = i % r
        x = x * g % p
    return colors[1:]

def max_run(cols):
    best = cur = 1
    for a, b in zip(cols, cols[1:]):
        cur = cur + 1 if a == b else 1
        best = max(best, cur)
    return best

def leading_run(cols):
    n = 1
    while n < len(cols) and cols[n] == cols[0]:
        n += 1
    return n

def rabung_ok(cols, t):
    if max_run(cols) >= t: return False
    cap = t // 2 if cols[0] == cols[-1] else t - 1   # t//2 == ceil((t-1)/2)
    return leading_run(cols) < cap

def build_certificate(p, r, t, cols):
    cert = []
    for k in range(t - 1):
        cert.append(k % r)      # the multiple k*p
        cert.extend(cols)       # k*p+1 .. (k+1)*p-1
    cert.append((t - 1) % r)    # the last multiple, (t-1)*p
    return cert                 # colors of 0 .. (t-1)*p

false_valid = tested = passed = 0
primes = [q for q in range(5, 800)
          if all(q % d for d in range(2, int(q**.5) + 1))]
for p in primes:
    for r, t in ((2, 4), (2, 5), (2, 6), (3, 4), (3, 5)):
        if (p - 1) % r: continue
        cols = residue_colors(p, r)
        tested += 1
        if rabung_ok(cols, t):
            passed += 1
            if has_mono_ap(build_certificate(p, r, t, cols), t) is not None:
                false_valid += 1
assert false_valid == 0
print(f"criterion vs brute force: {tested} (p,r,t) cases, "
      f"{passed} criterion-valid, {false_valid} false positives")
print("the criterion NEVER over-claims on any tested case")

## Step 3 — Reproduce two *known* records, certificates brute-checked in full

Now the pipeline reproduces history. **W(2,7) > 3703** is Rabung's own 1979 record (still standing today!), from p = 617. **W(3,9) > 932,745** is a modern three-color record (p = 116,593, same family as our new bounds). In both cases we build the *entire* certificate and run the check-every-AP function on it — millions to billions of progressions, zero found.

In [ ]:
cols617 = residue_colors(617, 2)
assert rabung_ok(cols617, 7)
cert = build_certificate(617, 2, 7, cols617)
assert len(cert) == 6 * 617 + 1 == 3703
assert has_mono_ap(cert, 7) is None
print("W(2,7) > 3703 (Rabung 1979) reproduced; full certificate brute-checked")

def has_mono_ap_fast(cert, t):
    """Same check as has_mono_ap, vectorized for big certificates."""
    c = np.asarray(cert, dtype=np.uint8)
    n = len(c)
    for d in range(1, (n - 1) // (t - 1) + 1):
        m = n - (t - 1) * d
        eq = np.ones(m, dtype=bool)
        for j in range(1, t):
            np.logical_and(eq, c[j*d:j*d+m] == c[:m], out=eq)
            if not eq.any(): break
        else:
            return True
    return False

cols3 = residue_colors(116593, 3)
assert rabung_ok(cols3, 9)
assert not has_mono_ap_fast(build_certificate(116593, 3, 9, cols3), 9)
print("W(3,9) > 932,745 reproduced; full 932,745-entry certificate brute-checked")

## Step 4 — The record prime

p = 969,397,381 is too large for the pure-Python discrete-log walk, so the next cell computes the same coloring a different way: cls(n) = the cube-root-of-unity class of n^((p−1)/3) mod p, vectorized and chunked. **We first cross-check this implementation against the simple one** on p = 116,593 (they must agree on the run structure exactly), then run it on the record prime.

⏱️ *The final call takes a few minutes — it computes a modular exponentiation for each of ~10⁹ residues.*

The result to watch for: **max run = 18**, leading run = 1, boundary satisfied — which by Rabung's criterion (validated above) certifies **W(3,t) > (t−1)·969,397,381 + 1 for every t ≥ 19.**

In [ ]:
def residue_max_run_big(p, r, chunk=8_000_000):
    e = (p - 1) // r
    def is_gen(g):
        m, fac, d = p - 1, [], 2
        while d * d <= m:
            if m % d == 0:
                fac.append(d)
                while m % d == 0: m //= d
            d += 1
        if m > 1: fac.append(m)
        return all(pow(g, (p - 1) // q, p) != 1 for q in fac)
    g = next(g for g in range(2, p) if is_gen(g))
    zeta = pow(g, e, p)
    roots = np.array(sorted(pow(zeta, j, p) for j in range(r)), dtype=np.int64)
    lab = {int(v): j for j, v in enumerate(roots)}
    labs = np.array([lab[int(v)] for v in roots], dtype=np.uint8)
    bits = [int(b) for b in bin(e)[2:]]
    best = cur_len = 0; cur_val = -1; first = last = lead = None
    for lo in range(1, p, chunk):
        hi = min(lo + chunk, p)
        nvec = np.arange(lo, hi, dtype=np.int64)
        acc = np.ones(hi - lo, dtype=np.int64)
        for b in bits:
            acc = (acc * acc) % p
            if b: acc = (acc * nvec) % p
        pos = np.searchsorted(roots, acc)
        assert (roots[pos] == acc).all()
        cols = labs[pos]
        if first is None: first = int(cols[0])
        ch = np.nonzero(np.diff(cols))[0]
        if len(ch) == 0:
            cur_len = cur_len + len(cols) if int(cols[0]) == cur_val else len(cols)
            cur_val = int(cols[0]); best = max(best, cur_len)
        else:
            l0 = int(ch[0]) + 1
            best = max(best, cur_len + l0 if int(cols[0]) == cur_val else l0)
            if lead is None and lo == 1: lead = l0
            inner = np.diff(np.concatenate((ch, [len(cols) - 1])))
            if len(inner): best = max(best, int(inner.max()))
            cur_val = int(cols[-1]); cur_len = int(len(cols) - 1 - ch[-1])
        last = int(cols[-1])
    return max(best, cur_len), first, last, lead

mr, f, l, ld = residue_max_run_big(116593, 3, chunk=30000)
assert (mr, ld) == (max_run(cols3), leading_run(cols3))
print("chunked implementation agrees with the simple one on p=116,593")

P = 969_397_381
mr, f, l, ld = residue_max_run_big(P, 3)
print(f"record prime p={P}: max_run={mr}, leading_run={ld}")
assert mr == 18 and ld == 1 and ld < (19 // 2 if f == l else 18)
print()
for t in range(19, 26):
    print(f"  W(3,{t}) > {(t-1)*P + 1:,}")

## What a human still needs to check (the honest residue)

The computation above is now as verified as computation gets. What it does **not** establish by itself:

1. **That the previous bounds are what we say they are.** Check the r=3 column of Monroe's table: his paper ([JCMCC 128 (2026) 305–315](https://combinatorialpress.com/article/jcmcc/Volume%20128/new-lower-bounds-for-van-der-waerden-numbers-using-distributed-computing.pdf); [arXiv:1603.03301](https://arxiv.org/abs/1603.03301)) and the tally file `output.txt` in [github.com/hmonroe/vdw](https://github.com/hmonroe/vdw). His scan was exhaustive to ~969,396,749 and stopped in 2019 (the BOINC project died); our prime is the first valid one past that mark.
2. **That no recursion theorem shadows these cells.** For r ≥ 4, bounds implied by the Blankenship–Cummings–Taranchuk recursion ([EJC 69 (2018)](https://arxiv.org/abs/1705.09673)) dwarf any scan — we found (and discarded) three such shadowed "records" earlier the same day. For r = 3 at these lengths the recursion gives ~t·W(2,t) ≈ 10⁸–10¹¹·t… concretely, far below the scan values; and Xu-type recursions multiply *color counts*, giving nothing for prime r = 3. This is checkable arithmetic from the cited papers, and it is *why Monroe's own published tables include r = 3*.
3. **Community convention** — that extending a stopped exhaustive scan counts as the new best known. It does: it is precisely how the standing records were set and published, most recently December 2025.

**Recommended next step:** email Tim/Hugh Monroe (contact on the arXiv paper / GitHub repo) — the person who maintains exactly these tables, who published the standing values seven months ago, and who explicitly flagged the near-billion band as incompletely checked. The message is three sentences: *we continued your scan past 969,396,749; the attached notebook and prime list are the result; could you confirm?* Marijn Heule (CMU) is a natural second reader. This is a normal, welcome email in this community — certificate results are checked by recomputation, not refereeing.

## Post-mortem: the search strategy, honestly

This result came at the end of a one-day session that started with "pick a fun unsolved problem." The retrospective matters more than the records, because the *pattern* is reusable. The day, compressed:

1. **Ramsey numbers (classical cells: R(4,6), R(4,7), R(3,3,3,3)).** Simulated annealing over circulant then Cayley-graph colorings: reproduced every known extremal witness (Paley 17, the GF(16) 3-coloring, tight R(3,9)/R(4,5) colorings) and set nothing. Pivot: freezing a record graph and asking "does a one-vertex extension exist?" turns an unsearchable 2^630 space into an exactly-decidable 35-variable CSP. Result — a real one, though negative: **all 39 known record graphs for these cells, and every valid 1- and 2-edge modification of them (~700k variants), are provably non-extendable.** The records sit in deep isolated basins ≥ 3 edits from anything better; that is *why* they have stood for decades. An hour of stochastic search had flailed at what exact decision settled in milliseconds.
2. **The zipper trap.** We implemented the cyclic-zipper doubling method for van der Waerden bounds, validated it against every known zipped record (with full brute-force certificate checks) and against the reference C code — it caught a transcription bug via the paper's own worked example — and swept 66 never-checked candidate primes. Three passed: valid, novel certificates for W(8,23), W(6,24), W(8,21). **None are records**: a 2018 recursion theorem implies bounds 10⁵–10¹⁶× larger for r ≥ 4. Verify the *record-keeping*, not just the mathematics, before celebrating.
3. **The seam.** The same verification pass revealed where recursions do NOT reach: r = 2 and r = 3 at large t, where the standing records are literally "the largest prime an exhaustive scan reached before its distributed-computing project died in 2019." The r = 3 table cells sit at the scan cap — meaning valid primes are *dense* there — and nobody had checked a single prime past 969,396,749 in six years. The scanner found 14 valid primes in its first hour; roughly half of all primes checked were valid. The seven records fell out of the most trivial code written all day.

### What actually moved the needle

Every unit of progress came from moving the search **up a level**; none came from searching harder at the current level. The object-level search (annealing, modular exponentiation) was commodity. The outer loops — *which problem, which table, which cell, which method* — were driven by **reading**: papers, source code, tally files, and the sociology of which compute efforts had stopped and when. The decisive facts were things like a hard-coded `if (prime > 40000000) return 0;` in a 2016 C file, and a table caption saying "we only present bounds for r ≤ 4 since for more colors our bounds were beaten by the recursions." All of the alpha was in the field's metadata.

The reusable screening profile for "winnable by this kind of effort":
1. the claim is **certificate-checkable** (cheap verification is both the search's fitness function and its insurance policy);
2. records were set by **compute that visibly stopped** (dead BOINC project, 2013-era SAT solver, a 5-minute solver timeout in a 2017 paper) rather than by an idea;
3. **no theory shadow** — check the recursion/implication literature *first*;
4. an actively **maintained table** exists to land the result in.

Also for the record: the single biggest constant-factor win of the day (a ~100× early-abort restructuring of the candidate sweep) came from a human looking at a loop and asking whether it was really necessary to pay full price for candidates that fail in the first megabyte. The heuristic she cited was "never use a loop." The loop was innocent. The instinct was correct anyway.

## Appendix: provenance & artifacts

- All 14 valid primes and 78 record entries: `VDW_RECORDS.json` (this repo). The best-per-cell bounds all come from p = 969,397,381; the other 13 primes independently corroborate (each is its own certificate).
- Search/verification code as actually run: `code/` in this repo (`vdw.py`, `vdw_scan3.py`, `vdw_zip*.py` — the latter validated against R. Rabung & M. Lotts, *Improving the Use of Cyclic Zippers in Finding Lower Bounds for van der Waerden Numbers*, E-JC 19(2) #P35 (2012), and Monroe's reference implementation).
- Independent confirmation already performed: the record prime rescanned with a different primitive root (labels permute; run structure — max run 18, leading run 1 — reproduced exactly) plus 50,000-position scalar spot-checks of the vectorized arithmetic against Python's built-in `pow`.
- Session lab notebook with the full decision trail: `NOTES.md`.

*Prepared July 20, 2026. If you are a mathematician reading this because someone you once dated in grad school sent it to you: the authors apologize, and cell-by-cell execution takes about five minutes, most of it in Step 4.*